In [ ]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Set absolute path for new metastore
new_metastore_path = os.path.abspath('/apps/sandbox/metastore')
metastore_url = f"jdbc:derby:;databaseName={new_metastore_path};create=true"

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.sql.catalogImplementation", "hive")
    .set("spark.sql.catalog.spark_catalog.warehouse","s3a://warehouse/default")
    .set("spark.sql.warehouse.dir", "s3a://warehouse/default/spark")
    .set("javax.jdo.option.ConnectionURL", metastore_url)
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("09-liquid-clustering").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

In [ ]:
%load_ext sql

In [ ]:
%sql spark

In [0]:
df1 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet")
df2 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df3 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_201_99457.parquet")

In [0]:
df_union = df1.union(df2).union(df3)
df_union = df_union.select("customer_id", "category", "price", "quantity", "invoice_date")

In [0]:
df_union.write.mode("overwrite").partitionBy("invoice_date").saveAsTable("spark_catalog.deltalake.lc_ex1")

In [0]:
%%sql
OPTIMIZE spark_catalog.deltalake.lc_ex1
ZORDER BY customer_id; 

In [0]:
%%sql
SELECT count(*)
FROM FROM spark_catalog.deltalake.lc_ex1; 

FROM
99457


In [0]:
df_union.write.mode("overwrite").clusterBy("invoice_date", "customer_id").saveAsTable("spark_catalog.deltalake.lc_ex2")

In [0]:
%%sql
SELECT count(*)
FROM FROM spark_catalog.deltalake.lc_ex2; 

FROM
99457


In [0]:
%%time
spark.sql(
    """
    SELECT category,
        SUM(price * quantity) AS total_sales
    from spark_catalog.deltalake.lc_ex1
    WHERE (invoice_date BETWEEN '2021-01-01' AND '2023-12-31') AND 
        customer_id = 201
    GROUP BY category
    """
)

CPU times: user 2.03 ms, sys: 268 µs, total: 2.3 ms
Wall time: 145 ms


DataFrame[category: string, total_sales: double]

In [0]:
%%time
spark.sql(
    """
    SELECT category,
        SUM(price * quantity) AS total_sales
    from spark_catalog.deltalake.lc_ex2
    WHERE (invoice_date BETWEEN '2021-01-01' AND '2023-12-31') AND 
        customer_id = 201
    GROUP BY category
    """
)

CPU times: user 3.19 ms, sys: 509 µs, total: 3.7 ms
Wall time: 152 ms


DataFrame[category: string, total_sales: double]